# Dose-Response Curve Fitting with openfit

This notebook demonstrates fitting a **four-parameter logistic (4PL / Hill4P)** model to a simulated dose-response dataset using the `openfit` library.

## What is the 4PL model?

The four-parameter logistic (4PL) equation is:

$$y = \text{Bottom} + \frac{\text{Top} - \text{Bottom}}{1 + \left(\frac{\text{EC50}}{x}\right)^{\text{HillSlope}}}$$

| Parameter | Meaning |
|-----------|--------|
| **Bottom** | Lower asymptote (signal at zero concentration) |
| **Top** | Upper asymptote (signal at saturating concentration) |
| **EC50** | Concentration producing 50% of maximal effect |
| **HillSlope** | Slope factor (steepness of the sigmoidal transition) |

## Why use 1/y² weighting for ELISA-style data?

ELISA measurements exhibit approximately **constant coefficient of variation (CV)**, meaning the standard deviation scales proportionally with the signal. When variance is proportional to y², the appropriate weight is **1/y²**, which gives each point equal influence on the fit in relative (%) terms. Unweighted least-squares would over-weight high-signal points and under-weight low-signal points, leading to biased parameter estimates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from openfit import Fit

## Generate synthetic dose-response data

We simulate a 4PL curve with known ground-truth parameters:

- **Bottom** = 2 (baseline signal, e.g., background absorbance)
- **Top** = 98 (maximum signal at saturation)
- **EC50** = 10 (midpoint concentration)
- **HillSlope** = 1.5 (moderately steep sigmoid)

Multiplicative noise with ~15% CV is added to mimic a typical ELISA plate, where measurement variability is proportional to the signal level.

In [ ]:
rng = np.random.default_rng(42)
true_params = dict(Bottom=2.0, Top=98.0, EC50=10.0, HillSlope=1.5)
x = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0])
# True 4PL values
y_true = true_params["Bottom"] + (true_params["Top"] - true_params["Bottom"]) / (
    1 + (true_params["EC50"] / x) ** true_params["HillSlope"]
)
# Add ~15% CV multiplicative noise
y = y_true * (1 + rng.normal(0, 0.15, size=len(x)))
y = np.clip(y, 0.1, None)  # no negative signal
print("x:", x)
print("y:", np.round(y, 2))

## Fit the 4PL model

We pass the model name `"hill4p"`, the concentration array `x`, and the response array `y` to `Fit`, then call `.run()` to perform nonlinear least-squares optimization.

**Weight choice — `weights="1/y2"`:**  
Because ELISA CV is approximately constant (variance ∝ y²), the 1/y² weight scheme is the statistically correct choice. Each residual is divided by the measured response, so points at all signal levels contribute equally in percentage terms.

In [ ]:
result = Fit("hill4p", x, y, weights="1/y2").run()
print(result.summary())

## Interpret the results

The `result` object exposes the following fields:

| Field | Description |
|-------|-------------|
| `result.params` | Dict of fitted parameter values |
| `result.se` | Dict of parameter standard errors |
| `result.ci` | Dict of 95% asymptotic confidence intervals (lo, hi) |
| `result.r_squared` | Coefficient of determination (weighted) |
| `result.aicc` | Akaike Information Criterion, corrected for small samples |
| `result.rss` | Weighted residual sum of squares |
| `result.dof` | Degrees of freedom (n − p) |
| `result.standardized_residuals` | Residuals divided by their estimated SD |

A good fit shows: recovered parameters close to truth, narrow CIs, R² near 1, and standardized residuals randomly scattered between −2 and +2.

In [ ]:
print("Fitted parameters:")
for name, val in result.params.items():
    se = result.se[name]
    lo, hi = result.ci[name]
    print(f"  {name:12s} = {val:8.3f}  SE={se:.3f}  95%CI=[{lo:.3f}, {hi:.3f}]")

print(f"\nR\u00b2 = {result.r_squared:.4f}")
print(f"AICc = {result.aicc:.2f}")
print(f"RSS = {result.rss:.4f}")
print(f"DOF = {result.dof}")

## Plot the fit

Two panels are shown side by side:

- **Left:** The fitted 4PL curve overlaid on the raw data, with a vertical dashed line marking the EC50.
- **Right:** Standardized residuals vs. concentration on a log scale. Residuals should be randomly scattered around zero with no systematic curvature (which would indicate model misspecification) and most points within ±2.

In [ ]:
x_curve = np.logspace(np.log10(x.min()), np.log10(x.max()), 200)
# Evaluate model at fine grid
p = result.params
y_curve = p["Bottom"] + (p["Top"] - p["Bottom"]) / (1 + (p["EC50"] / x_curve) ** p["HillSlope"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: fit overlay
axes[0].semilogx(x_curve, y_curve, "b-", linewidth=2, label="4PL fit")
axes[0].semilogx(x, y, "ko", markersize=6, label="Data")
axes[0].axvline(p["EC50"], color="r", linestyle="--", alpha=0.7, label=f"EC50={p['EC50']:.2f}")
axes[0].set_xlabel("Concentration")
axes[0].set_ylabel("Response")
axes[0].set_title("4PL Fit")
axes[0].legend()

# Right: residuals
axes[1].semilogx(x, result.standardized_residuals, "ko", markersize=6)
axes[1].axhline(0, color="gray", linestyle="-")
axes[1].axhline(2, color="r", linestyle="--", alpha=0.5)
axes[1].axhline(-2, color="r", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Concentration")
axes[1].set_ylabel("Standardized Residual")
axes[1].set_title("Residuals")

plt.tight_layout()
plt.savefig("/tmp/dose_response_fit.png", dpi=100, bbox_inches="tight")
plt.show()
print("Residuals look random — no systematic pattern.")

## Save a report and reproducibility manifest

openfit can export two artifacts:

1. **HTML report** — a self-contained file with the fit plot, parameter table, diagnostic statistics, and the model equation.
2. **FitSpec JSON** — a lightweight reproducibility manifest that captures the model name, weight scheme, fitted parameters, software versions, and a hash of the input data. Anyone with this spec can reconstruct the exact fit without re-running the notebook.

In [ ]:
# HTML report (fit plot + parameter table + diagnostics)
result.report("/tmp/dose_response_report.html")
print("Report saved to /tmp/dose_response_report.html")

# FitSpec: the reproducibility manifest
result.spec.to_json("/tmp/dose_response.spec.json")
print("\nFitSpec saved. Contents:")
import json
spec_dict = json.loads(open("/tmp/dose_response.spec.json").read())
for k, v in spec_dict.items():
    print(f"  {k}: {v}")

## Summary

In this notebook we:

1. **Simulated** a realistic ELISA-style dose-response dataset using a known 4PL curve with 15% CV multiplicative noise.
2. **Fitted** the `hill4p` model with `openfit`, using 1/y² weighting appropriate for constant-CV data.
3. **Inspected** the results — recovered parameters, standard errors, 95% confidence intervals, R², and AICc.
4. **Visualized** the fit and standardized residuals to confirm model adequacy.
5. **Exported** an HTML report and a FitSpec reproducibility manifest.

The `FitSpec` JSON contains everything needed to reproduce this exact result — model, parameters, weight scheme, data hash, and software versions. See `05_reproducibility.ipynb` for a full reproducibility demo.

**Next notebooks:**
- `02_model_comparison.ipynb` — compare 4PL vs 5PL vs linear models using AICc
- `03_batch_fitting.ipynb` — fit an entire 96-well plate in one call
- `04_bootstrap_ci.ipynb` — bootstrap confidence intervals vs asymptotic CIs
- `05_reproducibility.ipynb` — replaying a fit from a FitSpec manifest